# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR² Mass Spectrometry Extracellular Vesicle Mast Cell dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset's Croissant schema is sourced from:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Let's load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # DatasetMetaData instance

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Let's review the available record sets, their `@id`s, fields, and columns in this dataset.


> **Note:** All entities in Croissant are uniquely referenced by their `@id` value.

In [ ]:
# Fetch and display record sets and fields by @id
print("Available Record Sets:")
for rset in meta.record_sets:
    print(f"  Record set name: {rset.name}\n    @id: {rset.id}")
    print("    Fields:")
    for field in rset.fields:
        print(f"      - {field.name}  (@id: {field.id}, dataType: {field.data_type})")


## 3. Data Extraction
Let's extract the records from one or more record sets into pandas DataFrames for exploration. Use the `@id` from the overview above.

> ⚡ **Tip:** You can programmatically retrieve all record set @ids and select which to analyze.

In [ ]:
# Identify all available record set ids
record_set_ids = [rset.id for rset in meta.record_sets]
print('RecordSet @ids:', record_set_ids)

# For demonstration, let’s select the primary RecordSet used for protein mass spec data
# (Replace the following @id with the desired one from the previous cell’s output)
chosen_record_set_id = record_set_ids[0] if record_set_ids else None
if chosen_record_set_id is None:
    raise ValueError("No record sets found in the dataset metadata.")

dataframes = {}
for rsid in record_set_ids:
    # Load all records for each record set
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records for RecordSet '@id': {rsid}")

# Show columns and preview data for the primary record set
print(f"\nColumns in RecordSet '@id': {chosen_record_set_id}")
print(dataframes[chosen_record_set_id].columns.tolist())
dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let’s conduct some data processing on one record set. We’ll:
1. Select a numeric field using its `@id` (such as abundance, peptide count, MW, etc.),
2. Filter records by value,
3. Normalize the numeric field,
4. Optionally group by a categorical field (e.g., protein description or accession).

In [ ]:
# Get the column list for our chosen record set
df = dataframes[chosen_record_set_id]
print("Sample columns:", df.columns[:10].tolist())

# Try to find likely numeric and group fields
potential_numeric_fields = [col for col in df.columns if any(keyword in col.lower() for keyword in ['abundance', 'count', 'mw', 'coverage', 'length', 'peptide', 'score'])]
print("Potential numeric fields:", potential_numeric_fields)
numeric_field = potential_numeric_fields[0] if potential_numeric_fields else df.select_dtypes('number').columns[0]

threshold = df[numeric_field].quantile(0.8)  # use top 20% as example
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (first 5 records):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try group by a likely attribute (description, accession, etc.)
group_fields = [col for col in df.columns if any(k in col.lower() for k in ['description', 'accession', 'sample'])]
group_field = group_fields[0] if group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (preview):")
    print(grouped_df.head())
else:
    print("No suitable group field detected.")

## 5. Visualization

Let's plot the distribution of the selected numeric field and examine its relationship with a grouping variable, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If a group field is present, show boxplot
if group_field:
    plt.figure(figsize=(12, 6))
    # For display, use only first 10 groups if too many
    top_groups = df[group_field].value_counts().index[:10]
    subset = df[df[group_field].isin(top_groups)]
    sns.boxplot(data=subset, x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
    plt.show()


## 6. Conclusion

- We loaded and reviewed the metadata of the FAIR² Extracellular Vesicle Mass Spec dataset using only `@id`-based references.
- Explored available record sets and their fields.
- Extracted records into DataFrames and performed initial filtering, normalization, and grouping on numeric protein/protein attribute columns.
- Visualized key distributions to start understanding the structure of the data.

> Try exploring other record sets or analyzing the data using alternative field combinations identified in Step 2! See the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) for advanced usage.